In [13]:
!pip install openai azure-core


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
import os
from openai import AzureOpenAI

# 1. Base Endpoint URL (strip any '/openai/v1' suffix to avoid 404 errors)
raw_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "")
AZURE_OPENAI_ENDPOINT = raw_endpoint.split("/openai")[0].rstrip("/")

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-5")
AZURE_OPENAI_API_VERSION = "2024-06-01"

# Validate configuration
if AZURE_OPENAI_API_KEY.startswith("BKStjfb"):
    print(" WARNING: Default API Key detected. Update AZURE_OPENAI_API_KEY with your Azure Portal credentials.")

# 2. Initialize the Azure OpenAI Client
client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

print(" Azure OpenAI Client Initialized Successfully!")
print(f"   Endpoint: {AZURE_OPENAI_ENDPOINT}")
print(f"   Deployment Target: {AZURE_OPENAI_DEPLOYMENT}")

 Azure OpenAI Client Initialized Successfully!
   Endpoint: https://azure-foundry-08.openai.azure.com
   Deployment Target: gpt-5


Defining Settlement Domain Schemas & State Objects

In [15]:
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
from datetime import datetime

@dataclass
class SettlementLineItem:
    order_id: str
    sku: str
    transaction_type: str # Order, Refund, Service Fee
    amount: float
    fba_fulfillment_fee: float
    commission_fee: float
    promotional_discount: float

@dataclass
class ReconciliationState:
    settlement_id: str
    raw_payload: str
    plan: List[str] = field(default_factory=list)
    current_step: int = 0
    extracted_items: List[SettlementLineItem] = field(default_factory=list)
    erp_matched_invoices: Dict[str, Any] = field(default_factory=dict)
    variances_detected: List[Dict[str, Any]] = field(default_factory=list)
    requires_hitl: bool = False
    status: str = "INITIALIZED"
    execution_log: List[str] = field(default_factory=list)

    def log(self, message: str):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        entry = f"[{timestamp}] {message}"
        self.execution_log.append(entry)
        print(entry)

# Test State Initialization with sample Amazon Settlement Header
sample_raw_csv = """Settlement ID: SET-AZN-2026-8891
Order-ID,SKU,Type,Amount,FBA-Fee,Commission,Promo
112-40918-01,NW-WIDGET-01,Order,150.00,-12.50,-22.50,-5.00
112-40918-02,NW-GADGET-02,Order,85.00,-8.00,-12.75,0.00
112-40918-03,NW-WIDGET-01,Refund,-150.00,0.00,22.50,0.00"""

state = ReconciliationState(
    settlement_id="SET-AZN-2026-8891",
    raw_payload=sample_raw_csv
)
state.log("Reconciliation state graph initialized for SET-AZN-2026-8891.")

[2026-08-17 06:13:10] Reconciliation state graph initialized for SET-AZN-2026-8891.


In [16]:
import json

class SettlementPlannerAgent:
    def __init__(self, client: AzureOpenAI, deployment: str):
        self.client = client
        self.deployment = deployment
        self.system_prompt = (
            "You are the Planner/Supervisor Agent for Northwind Global Retail's Amazon Reconciliation Engine.\n"
            "Your job is to analyze incoming raw settlement file payloads and generate a step-by-step execution plan.\n"
            "Always return your response as a valid JSON array of strings representing the task sequence."
        )

    def generate_plan(self, state: ReconciliationState) -> List[str]: #Method Workflow
        state.log("Planner Agent evaluating raw settlement payload...")
        
        user_prompt = f"Generate an execution plan for processing this Amazon Settlement file:\n{state.raw_payload}"
        
        response = self.client.chat.completions.create(
            model=self.deployment,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"}
        )
        
        raw_json = response.choices[0].message.content
        try:
            parsed = json.loads(raw_json)
            # Support flexible JSON keys from model response
            plan_steps = parsed.get("plan", parsed.get("steps", list(parsed.values())[0]))
            state.plan = plan_steps
            state.status = "PLANNED"
            state.log(f"Planner Agent generated {len(plan_steps)} steps.")
            return plan_steps
        except Exception as e:
            state.log(f" Failed to parse Planner output: {e}")
            # Fallback default plan aligned with Capstone pipeline
            fallback = [
                "1. Parse raw CSV into structured line items.",
                "2. Extract FBA fees, commission, and promotional discounts.",
                "3. Perform 3-way matching against Dynamics 365 ERP open invoices.",
                "4. Flag variances exceeding policy thresholds for HITL review."
            ]
            state.plan = fallback
            return fallback

# Instantiate and execute Planner Agent
planner = SettlementPlannerAgent(client, AZURE_OPENAI_DEPLOYMENT)
steps = planner.generate_plan(state)

print("\n--- GENERATED EXECUTION PLAN ---")
for idx, step in enumerate(state.plan, 1):
    print(f"{idx}. {step}")

[2026-08-17 06:13:10] Planner Agent evaluating raw settlement payload...


[2026-08-17 06:13:49] Planner Agent generated 31 steps.

--- GENERATED EXECUTION PLAN ---
1. Initialize reconciliation run and ensure idempotency for Settlement ID SET-AZN-2026-8891 (abort if already processed).
2. Persist the raw settlement payload to object storage with metadata (source=Amazon, settlement_id=SET-AZN-2026-8891, received_at=now).
3. Parse the CSV into structured rows using the provided header (Order-ID,SKU,Type,Amount,FBA-Fee,Commission,Promo); trim whitespace and cast numeric fields to decimal with 2 places.
4. Validate schema and data quality: required columns present; Types ∈ {Order, Refund}; numeric fields are valid; no NaN/blank critical fields; enforce USD default currency.
5. Normalize identifiers: derive base_order_id by joining the first two hyphen-delimited segments of Order-ID (e.g., 112-40918 from 112-40918-01/02/03); keep full line_id as Order-ID.
6. Standardize transaction types: map Type=Order → SALE; Type=Refund → REFUND; tag each row with internal txn_

In [17]:
class ReActExtractionAgent:
    def __init__(self, client: AzureOpenAI, deployment: str):
        self.client = client
        self.deployment = deployment

    def execute_extraction(self, state: ReconciliationState):
        state.log("ReAct Extraction Agent executing Step 1 & 2: Line-Item Parsing...")
        
        lines = [line.strip() for line in state.raw_payload.strip().split("\n") if line.strip() and not line.startswith("Settlement ID")]
        headers = lines[0].split(",")
        
        extracted_items = []
        for row in lines[1:]:
            parts = row.split(",")
            if len(parts) >= 7:
                item = SettlementLineItem(
                    order_id=parts[0],
                    sku=parts[1],
                    transaction_type=parts[2],
                    amount=float(parts[3]),
                    fba_fulfillment_fee=float(parts[4]),
                    commission_fee=float(parts[5]),
                    promotional_discount=float(parts[6])
                )
                extracted_items.append(item)
                state.log(f"Extracted Order {item.order_id} | Amount: ${item.amount:.2f} | FBA Fee: ${item.fba_fulfillment_fee:.2f}")

        state.extracted_items = extracted_items
        state.current_step = 2
        state.status = "EXTRACTED"
        state.log(f"ReAct Extraction Agent completed. Total Items: {len(extracted_items)}")

# Instantiate and run Extraction Agent
extractor = ReActExtractionAgent(client, AZURE_OPENAI_DEPLOYMENT)
extractor.execute_extraction(state)

[2026-08-17 06:13:49] ReAct Extraction Agent executing Step 1 & 2: Line-Item Parsing...
[2026-08-17 06:13:49] Extracted Order 112-40918-01 | Amount: $150.00 | FBA Fee: $-12.50
[2026-08-17 06:13:49] Extracted Order 112-40918-02 | Amount: $85.00 | FBA Fee: $-8.00
[2026-08-17 06:13:49] Extracted Order 112-40918-03 | Amount: $-150.00 | FBA Fee: $0.00
[2026-08-17 06:13:49] ReAct Extraction Agent completed. Total Items: 3


In [18]:
# Simulated Dynamics 365 ERP Open Invoices Ledger
MOCK_D365_ERP_INVOICES = {
    "112-40918-01": {"invoice_id": "INV-D365-9901", "expected_net": 110.00, "status": "OPEN"},
    "112-40918-02": {"invoice_id": "INV-D365-9902", "expected_net": 70.00, "status": "OPEN"},
    "112-40918-03": {"invoice_id": "INV-D365-9903", "expected_net": -127.50, "status": "OPEN"}
}

class ReActMatchingAgent:
    def __init__(self):
        pass

    def perform_3way_matching(self, state: ReconciliationState):
        state.log("ReAct Matching Agent executing Step 3: D365 ERP 3-Way Matching...")
        
        for item in state.extracted_items:
            net_payout = item.amount + item.fba_fulfillment_fee + item.commission_fee + item.promotional_discount
            
            erp_record = MOCK_D365_ERP_INVOICES.get(item.order_id)
            if erp_record:
                expected = erp_record["expected_net"]
                variance = round(abs(expected - net_payout), 2)
                
                state.erp_matched_invoices[item.order_id] = {
                    "invoice_id": erp_record["invoice_id"],
                    "net_payout": net_payout,
                    "expected_net": expected,
                    "variance": variance
                }
                
                if variance > 0:
                    var_detail = {
                        "order_id": item.order_id,
                        "variance_amount": variance,
                        "reason": "FBA fee discrepancy or unallocated promotional discount"
                    }
                    state.variances_detected.append(var_detail)
                    state.log(f" Variance detected for Order {item.order_id}: ${variance:.2f}")
                else:
                    state.log(f" Order {item.order_id} matched perfectly with D365 {erp_record['invoice_id']}.")
            else:
                state.log(f" Order {item.order_id} NOT found in D365 ERP.")

        state.current_step = 3
        state.status = "MATCHED"

# Instantiate and run Matching Agent
matcher = ReActMatchingAgent()
matcher.perform_3way_matching(state)

[2026-08-17 06:13:49] ReAct Matching Agent executing Step 3: D365 ERP 3-Way Matching...
[2026-08-17 06:13:49]  Order 112-40918-01 matched perfectly with D365 INV-D365-9901.
[2026-08-17 06:13:49]  Variance detected for Order 112-40918-02: $5.75
[2026-08-17 06:13:49]  Order 112-40918-03 matched perfectly with D365 INV-D365-9903.


In [19]:
print("=========================================================================")
print(f"       RECONCILIATION REPORT: SETTLEMENT {state.settlement_id}")
print("=========================================================================")
print(f"📊 Final State Status: {state.status}")
print(f"📝 Execution Steps Completed: {state.current_step}/{len(state.plan)}")
print(f"📦 Extracted Line Items: {len(state.extracted_items)}")
print(f"🔗 ERP Invoices Matched: {len(state.erp_matched_invoices)}")
print(f"⚠️ Total Variances Flagged: {len(state.variances_detected)}")

print("\n--- DETECTED VARIANCES ---")
for var in state.variances_detected:
    print(f"• Order: {var['order_id']} | Discrepancy: ${var['variance_amount']:.2f} | Note: {var['reason']}")

print("\n--- CHRONOLOGICAL AUDIT LOG ---")
for log_entry in state.execution_log:
    print(log_entry)
print("=========================================================================")

       RECONCILIATION REPORT: SETTLEMENT SET-AZN-2026-8891
📊 Final State Status: MATCHED
📝 Execution Steps Completed: 3/31
📦 Extracted Line Items: 3
🔗 ERP Invoices Matched: 3
⚠️ Total Variances Flagged: 1

--- DETECTED VARIANCES ---
• Order: 112-40918-02 | Discrepancy: $5.75 | Note: FBA fee discrepancy or unallocated promotional discount

--- CHRONOLOGICAL AUDIT LOG ---
[2026-08-17 06:13:10] Reconciliation state graph initialized for SET-AZN-2026-8891.
[2026-08-17 06:13:10] Planner Agent evaluating raw settlement payload...
[2026-08-17 06:13:49] Planner Agent generated 31 steps.
[2026-08-17 06:13:49] ReAct Extraction Agent executing Step 1 & 2: Line-Item Parsing...
[2026-08-17 06:13:49] Extracted Order 112-40918-01 | Amount: $150.00 | FBA Fee: $-12.50
[2026-08-17 06:13:49] Extracted Order 112-40918-02 | Amount: $85.00 | FBA Fee: $-8.00
[2026-08-17 06:13:49] Extracted Order 112-40918-03 | Amount: $-150.00 | FBA Fee: $0.00
[2026-08-17 06:13:49] ReAct Extraction Agent completed. Total Ite

In [ ]:
import os
from openai import AzureOpenAI

# 1. Base Endpoint URL (strip any '/openai/v1' suffix to avoid 404 errors)
raw_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "")
AZURE_OPENAI_ENDPOINT = raw_endpoint.split("/openai")[0].rstrip("/")

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-5")
AZURE_OPENAI_API_VERSION = "2024-06-01"

# Validate configuration
if AZURE_OPENAI_API_KEY.startswith(""):
    print(" WARNING: Default API Key detected. Update AZURE_OPENAI_API_KEY with your Azure Portal credentials.")

# 2. Initialize the Azure OpenAI Client
client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

print("✅ Azure OpenAI Client Initialized Successfully!")
print(f"   Endpoint: {AZURE_OPENAI_ENDPOINT}")
print(f"   Deployment Target: {AZURE_OPENAI_DEPLOYMENT}")

✅ Azure OpenAI Client Initialized Successfully!
   Endpoint: https://azure-foundry-08.openai.azure.com
   Deployment Target: gpt-5


In [21]:
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any

# Pydantic Schema for Querying Dynamics 365 ERP
class D365InvoiceQuerySchema(BaseModel):
    order_id: str = Field(description="Amazon Seller Central Order ID (e.g., 112-40918-01)")
    asin: Optional[str] = Field(default=None, description="Amazon Standard Identification Number (ASIN)")

# Pydantic Schema for ERP Invoice Result
class D365InvoiceResponseSchema(BaseModel):
    invoice_id: str
    order_id: str
    sku: str
    customer_id: str
    gross_amount: float
    expected_fba_fee: float
    expected_net_payout: float
    payment_status: str

# Standardized MCP Tool Schema Registration (OpenAI Compatible Format)
MCP_D365_SEARCH_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "search_d365_erp_invoice",
        "description": "Exposes Microsoft Dynamics 365 ERP invoice lookup endpoint via Model Context Protocol (MCP) contract.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "Amazon Seller Central Order ID (e.g., 112-40918-01)"
                },
                "asin": {
                    "type": "string",
                    "description": "Optional ASIN filter for item-level invoice reconciliation"
                }
            },
            "required": ["order_id"]
        }
    }
}

print("✅ MCP Tool & Payload Schemas Defined Successfully!")
print(f"📋 Registered Tool: {MCP_D365_SEARCH_TOOL_SCHEMA['function']['name']}")

✅ MCP Tool & Payload Schemas Defined Successfully!
📋 Registered Tool: search_d365_erp_invoice


In [22]:
import json

class LocalD365ErpMcpServer:
    """Mock MCP Server exposing Microsoft Dynamics 365 ERP endpoints."""
    
    def __init__(self):
        # MOCKED D365 ERP DATABASE LEDGER
        self._database = {
            "112-40918-01": {
                "invoice_id": "INV-D365-2026-001",
                "order_id": "112-40918-01",
                "sku": "NW-WIDGET-01",
                "customer_id": "CUST-AZN-US",
                "gross_amount": 150.00,
                "expected_fba_fee": -12.50,
                "expected_net_payout": 110.00,
                "payment_status": "UNRECONCILED"
            },
            "112-40918-02": {
                "invoice_id": "INV-D365-2026-002",
                "order_id": "112-40918-02",
                "sku": "NW-GADGET-02",
                "customer_id": "CUST-AZN-US",
                "gross_amount": 85.00,
                "expected_fba_fee": -8.00,
                "expected_net_payout": 70.00,
                "payment_status": "UNRECONCILED"
            }
        }

    def handle_tool_call(self, tool_name: str, arguments_json: str) -> str:
        """Processes tool execution requests following MCP contract conventions."""
        if tool_name != "search_d365_erp_invoice":
            return json.dumps({"error": f"Unknown MCP tool: {tool_name}"})
        
        try:
            args = json.loads(arguments_json)
            order_id = args.get("order_id")
            
            print(f"⚙️ [MCP Server] Invoking endpoint 'search_d365_erp_invoice' with args: {args}")
            
            record = self._database.get(order_id)
            if record:
                return json.dumps({"status": "SUCCESS", "data": record})
            else:
                return json.dumps({"status": "NOT_FOUND", "message": f"Invoice for Order {order_id} not found in D365 ERP."})
        except Exception as e:
            return json.dumps({"status": "ERROR", "message": str(e)})

# Instantiate local MCP Server
mcp_server = LocalD365ErpMcpServer()

# Test direct MCP invocation
sample_test_response = mcp_server.handle_tool_call(
    "search_d365_erp_invoice", 
    json.dumps({"order_id": "112-40918-01"})
)
print(f"\n🔍 [MCP Server Direct Output]:\n{sample_test_response}")

⚙️ [MCP Server] Invoking endpoint 'search_d365_erp_invoice' with args: {'order_id': '112-40918-01'}

🔍 [MCP Server Direct Output]:
{"status": "SUCCESS", "data": {"invoice_id": "INV-D365-2026-001", "order_id": "112-40918-01", "sku": "NW-WIDGET-01", "customer_id": "CUST-AZN-US", "gross_amount": 150.0, "expected_fba_fee": -12.5, "expected_net_payout": 110.0, "payment_status": "UNRECONCILED"}}


In [23]:
class ReActMcpExecutorAgent:
    def __init__(self, client: AzureOpenAI, deployment: str, mcp_server: LocalD365ErpMcpServer):
        self.client = client
        self.deployment = deployment
        self.mcp_server = mcp_server
        self.system_prompt = (
            "You are the ReAct Matching Agent for Northwind Global Retail.\n"
            "Your objective is to reconcile Amazon settlement line items against Microsoft Dynamics 365 ERP invoices.\n"
            "Use the provided MCP tool 'search_d365_erp_invoice' to fetch expected ledger values before calculating financial variances."
        )

    def reconcile_order(self, order_id: str, amazon_net_payout: float) -> str:
        print(f"\n==================================================")
        print(f"🤖 [ReAct Agent] Processing Order: {order_id}")
        print(f"==================================================")
        
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"Reconcile Amazon Order ID '{order_id}' which has a reported net payout of ${amazon_net_payout:.2f}."}
        ]
        
        # Turn 1: Initial Model reasoning & potential tool call generation
        response = self.client.chat.completions.create(
            model=self.deployment,
            messages=messages,
            tools=[MCP_D365_SEARCH_TOOL_SCHEMA],
            tool_choice="auto"
        )
        
        response_message = response.choices[0].message
        messages.append(response_message)
        
        # Check if the model generated a tool call
        if response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = tool_call.function.arguments
                
                print(f"👉 [Agent Tool Request] Tool: '{function_name}' | Arguments: {function_args}")
                
                # Execute tool against local MCP server
                mcp_result = self.mcp_server.handle_tool_call(function_name, function_args)
                print(f"📥 [MCP Response Received]: {mcp_result}")
                
                # Append tool result back to conversation context
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": mcp_result
                })
            
            # Turn 2: Final reasoning turn with tool results included
            final_response = self.client.chat.completions.create(
                model=self.deployment,
                messages=messages
            )
            final_text = final_response.choices[0].message.content
            print(f"\n💡 [Agent Final Verdict]:\n{final_text}")
            return final_text
        else:
            print(f"⚠️ Agent did not invoke MCP tool. Response: {response_message.content}")
            return response_message.content

# Instantiate ReAct MCP Agent
react_mcp_agent = ReActMcpExecutorAgent(client, AZURE_OPENAI_DEPLOYMENT, mcp_server)

In [24]:
# Sample extracted settlement records from Lab 1.1 pipeline
sample_extracted_settlements = [
    {"order_id": "112-40918-01", "net_payout": 110.00}, # Perfect match ($150 - $12.50 FBA - $22.50 Comm - $5 Promo)
    {"order_id": "112-40918-02", "net_payout": 64.25}  # Discrepancy ($85 - $8 FBA - $12.75 Comm vs expected $70 net)
]

# Run reconciliation for each extracted order
reconciliation_results = []
for record in sample_extracted_settlements:
    verdict = react_mcp_agent.reconcile_order(
        order_id=record["order_id"],
        amazon_net_payout=record["net_payout"]
    )
    reconciliation_results.append({
        "order_id": record["order_id"],
        "reported_net": record["net_payout"],
        "summary": verdict
    })
    


🤖 [ReAct Agent] Processing Order: 112-40918-01


👉 [Agent Tool Request] Tool: 'search_d365_erp_invoice' | Arguments: {"order_id":"112-40918-01"}
⚙️ [MCP Server] Invoking endpoint 'search_d365_erp_invoice' with args: {'order_id': '112-40918-01'}
📥 [MCP Response Received]: {"status": "SUCCESS", "data": {"invoice_id": "INV-D365-2026-001", "order_id": "112-40918-01", "sku": "NW-WIDGET-01", "customer_id": "CUST-AZN-US", "gross_amount": 150.0, "expected_fba_fee": -12.5, "expected_net_payout": 110.0, "payment_status": "UNRECONCILED"}}

💡 [Agent Final Verdict]:
Here’s the reconciliation for Amazon Order ID 112-40918-01:

- D365 invoice matched: INV-D365-2026-001 (SKU: NW-WIDGET-01, Customer: CUST-AZN-US)
- D365 expected values:
  - Gross amount: $150.00
  - Expected FBA fee: -$12.50
  - Expected net payout: $110.00
  - Current payment status: UNRECONCILED
- Amazon reported net payout: $110.00

Variance analysis:
- Reported net payout vs D365 expected net payout: $110.00 − $110.00 = $0.00
- Result: No variance. The Amazon payout aligns with D